In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib


# ==========================================
# 1. CHARGEMENT ET PRÉPARATION
# ==========================================
print("1/6 - Chargement des données...")

# Chargement du fichier (vérifie bien le nom du fichier sur ton Colab)
df = pd.read_csv('data_prod_finale.csv', parse_dates=['time'], index_col='time')
df = df.sort_index()


1/6 - Chargement des données...


In [2]:
df.head()

,Niveau_reservoir,DECISION_BACKWASH_NUM,DP_moyen_filtres,type_backwash,duree_backwash,intervalle_min,freq_backwash,INDICE_ENCRASSEMENT_HMMF,DEBIT_ENTREE_HMMF,INDICE_IMPACT_PRODUCTION,...,HMMF-B,HMMF-C,HMMF-D,HMMF-E,HMMF-F,HMMF-G,HMMF-H,HMMF-I,HMMF-J,target
time,,,,,,,,,,,,,,,,,,,,,
2024-02-13 11:38:00,82.893753,3,0.695475,3,271.0,NaN,734.0,0.695475,415.330942,0.001675,...,1,1,1,1,1,1,1,1,1,1437.231247
2024-02-13 11:39:00,82.375000,3,0.694425,3,271.0,1.0,734.0,0.694425,414.367148,0.001676,...,1,1,1,1,1,1,1,1,1,1441.015625
2024-02-13 11:40:00,81.356247,3,0.693700,3,271.0,1.0,734.0,0.693700,413.566388,0.001677,...,1,1,1,1,1,1,1,1,1,1440.490616
2024-02-13 11:41:00,81.487503,3,0.692450,3,271.0,1.0,734.0,0.692450,412.985098,0.001677,...,1,1,1,1,1,1,1,1,1,1440.512512
2024-02-13 11:42:00,81.237503,3,0.693088,3,271.0,1.0,734.0,0.693088,413.029889,0.001678,...,1,1,1,1,1,1,1,1,1,1440.774994


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 449002 entries, 2024-02-13 11:38:00 to 2024-12-23 06:59:00
Data columns (total 32 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   Niveau_reservoir                       449002 non-null  float64
 1   DECISION_BACKWASH_NUM                  449002 non-null  int64  
 2   DP_moyen_filtres                       449002 non-null  float64
 3   type_backwash                          449002 non-null  int64  
 4   duree_backwash                         449002 non-null  float64
 5   intervalle_min                         449001 non-null  float64
 6   freq_backwash                          449002 non-null  float64
 7   INDICE_ENCRASSEMENT_HMMF               449002 non-null  float64
 8   DEBIT_ENTREE_HMMF                      449002 non-null  float64
 9   INDICE_IMPACT_PRODUCTION               449002 non-null  float64
 10  INDICE_DESEQUILIBRE_FI

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- 1. CHARGEMENT ET PRÉPARATION ---
print("Chargement des données...")
df1 = pd.read_csv('data_evolution/data_ready_to_use.csv')

target_col = "LIT_002_FILTER_WATER_STG_TNK"


Chargement des données...


In [8]:
df1.head()

,INTAKE LINE PRESSURE OF HMMF FEED PUMPS,FIT_001A_FLOW AT INLET HMMF-A,FIT_001B_FLOW AT INLET HMMF-B,FIT_001C_FLOW AT INLET HMMF-C,FIT_001D_FLOW AT INLET HMMF-D,FIT_001E_FLOW AT INLET HMMF-E,FIT_001F_FLOW AT INLET HMMF-F,FIT_001G_FLOW AT INLET HMMF-G,FIT_001H_FLOW AT INLET HMMF-H,FIT_001I_FLOW AT INLET HMMF-I,...,PIT_008G_Reject line pressure,PIT_008H_Reject line pressure,FIT005A_PERMEATE FLOW,FIT_005B_PERMEATE FLOW,FIT_005C_PERMEATE FLOW,FIT_005D_PERMEATE FLOW,FIT_005E_PERMEATE FLOW,FIT_005F_PERMEATE FLOW,FIT_005G_PERMEATE FLOW,FIT_005H_PERMEATE FLOW
0,1.49075,454.837494,391.059570,494.862335,388.456238,371.231262,452.521881,391.543762,452.603119,403.975006,...,58.3125,0.0,169.553131,189.918747,163.909378,184.274994,171.084381,195.278122,198.515625,164.696869
1,1.49650,454.837494,388.655182,493.031921,388.496887,369.078125,452.521881,391.421875,451.587494,403.812500,...,58.3125,0.0,169.312500,189.350006,163.865631,184.274994,171.718750,196.524994,198.515625,167.453125
2,1.49525,454.837494,386.984314,493.642059,388.984375,368.468750,452.521881,389.715637,449.596863,401.821869,...,58.3125,0.0,169.684372,191.756256,163.953125,184.274994,170.406250,196.153122,198.515625,165.746872
3,1.49625,454.837494,386.413788,492.259064,386.181244,367.981262,452.521881,390.040619,450.165619,400.968750,...,58.3125,0.0,169.443756,189.175003,163.274994,184.274994,170.909378,197.115631,198.515625,167.803131
4,1.50775,454.837494,387.391846,490.632050,387.968750,370.012512,452.521881,388.862488,450.409363,400.359375,...,58.3125,0.0,168.524994,190.771881,163.843750,184.274994,171.434372,195.409378,198.515625,168.000000


In [11]:
# Renommer la colonne dans df2
df1 = df1.rename(columns={'LIT_002_FILTER_WATER_STG_TNK':'Niveau_reservoir'})

# Vérification
print(df1.columns)


Index(['INTAKE LINE PRESSURE OF HMMF FEED PUMPS',
       'FIT_001A_FLOW AT INLET HMMF-A', 'FIT_001B_FLOW AT INLET HMMF-B',
       'FIT_001C_FLOW AT INLET HMMF-C', 'FIT_001D_FLOW AT INLET HMMF-D',
       'FIT_001E_FLOW AT INLET HMMF-E', 'FIT_001F_FLOW AT INLET HMMF-F',
       'FIT_001G_FLOW AT INLET HMMF-G', 'FIT_001H_FLOW AT INLET HMMF-H',
       'FIT_001I_FLOW AT INLET HMMF-I', 'FIT_001J_FLOW AT INLET HMMF-J',
       'PDIT_001A_DIFF PRESSURE ACROSS HMMF-A',
       'PDIT_001B_DIFF PRESSURE ACROSS HMMF-B',
       'PDIT_001C_DIFF PRESSURE ACROSS HMMF-C',
       'PDIT_001D_DIFF PRESSURE ACROSS HMMF-D',
       'PDIT_001E_DIFF PRESSURE ACROSS HMMF-E',
       'PDIT_001F_DIFF PRESSURE ACROSS HMMF-F',
       'PDIT_001G_DIFF PRESSURE ACROSS HMMF-G',
       'PDIT_001H_DIFF PRESSURE ACROSS HMMF-H',
       'PDIT_001I_DIFF PRESSURE ACROSS HMMF-I',
       'PDIT_001J_DIFF PRESSURE ACROSS HMMF-J',
       'PIT_001 DISCHARGE PRESSURE AT HMMF A-E',
       'PIT_001 DISCHARGE PRESSURE AT HMMF F-J', 'Niveau

In [13]:
cols_to_drop = [
    'FIT_005B_PERMEATE FLOW', 'FIT_005C_PERMEATE FLOW', 'FIT_005D_PERMEATE FLOW',
    'FIT_005E_PERMEATE FLOW', 'FIT_005F_PERMEATE FLOW', 'FIT_005G_PERMEATE FLOW',
    'FIT_005H_PERMEATE FLOW', 'FIT005A_PERMEATE FLOW'
]

df1 = df1.drop(columns=cols_to_drop)

# Vérification
print(df1.columns)


Index(['INTAKE LINE PRESSURE OF HMMF FEED PUMPS',
       'FIT_001A_FLOW AT INLET HMMF-A', 'FIT_001B_FLOW AT INLET HMMF-B',
       'FIT_001C_FLOW AT INLET HMMF-C', 'FIT_001D_FLOW AT INLET HMMF-D',
       'FIT_001E_FLOW AT INLET HMMF-E', 'FIT_001F_FLOW AT INLET HMMF-F',
       'FIT_001G_FLOW AT INLET HMMF-G', 'FIT_001H_FLOW AT INLET HMMF-H',
       'FIT_001I_FLOW AT INLET HMMF-I', 'FIT_001J_FLOW AT INLET HMMF-J',
       'PDIT_001A_DIFF PRESSURE ACROSS HMMF-A',
       'PDIT_001B_DIFF PRESSURE ACROSS HMMF-B',
       'PDIT_001C_DIFF PRESSURE ACROSS HMMF-C',
       'PDIT_001D_DIFF PRESSURE ACROSS HMMF-D',
       'PDIT_001E_DIFF PRESSURE ACROSS HMMF-E',
       'PDIT_001F_DIFF PRESSURE ACROSS HMMF-F',
       'PDIT_001G_DIFF PRESSURE ACROSS HMMF-G',
       'PDIT_001H_DIFF PRESSURE ACROSS HMMF-H',
       'PDIT_001I_DIFF PRESSURE ACROSS HMMF-I',
       'PDIT_001J_DIFF PRESSURE ACROSS HMMF-J',
       'PIT_001 DISCHARGE PRESSURE AT HMMF A-E',
       'PIT_001 DISCHARGE PRESSURE AT HMMF F-J', 'Niveau

In [15]:
# Colonnes uniques à df1
cols_unique_df1 = df1.columns.difference(df.columns)
print("Colonnes présentes uniquement dans df1 :", cols_unique_df1.tolist())

# Colonnes uniques à df2
cols_unique_df = df.columns.difference(df1.columns)
print("Colonnes présentes uniquement dans df2 :", cols_unique_df.tolist())


Colonnes présentes uniquement dans df1 : ['AIT007_PH at discharge HMMF FEED PUMPS', 'AIT_001A_PH at RO FEEDPUMP_P2-A/B/C/D', 'AIT_001B_PH at RO FEEDPUMP_P2-E/F/G/H', 'AIT_002A_ORP at RO FEEDPUMP_P2-A/B/C/D', 'AIT_002B_ORP at RO FEEDPUMP_P2-E/F/G/H', 'AIT_003A conductivity at RO FEEDPUMP_P2-A/B/C/D', 'AIT_003B conductivity at RO FEEDPUMP_P2-E/F/G/H', 'AIT_004A_COND', 'AIT_004B_COND', 'AIT_004C_COND', 'AIT_004D_COND', 'AIT_004E_COND', 'AIT_004F_COND', 'AIT_004G_COND', 'AIT_004H_COND', 'AIT_005A_PH', 'AIT_005B_PH', 'AIT_005C_PH', 'AIT_005D_PH', 'AIT_005E_PH', 'AIT_005F_PH', 'AIT_005G_PH', 'AIT_005H_PH', 'BACKWASH PUMP FLOW', 'FIT_001A_FLOW AT INLET HMMF-A', 'FIT_001B_FLOW AT INLET HMMF-B', 'FIT_001C_FLOW AT INLET HMMF-C', 'FIT_001D_FLOW AT INLET HMMF-D', 'FIT_001E_FLOW AT INLET HMMF-E', 'FIT_001F_FLOW AT INLET HMMF-F', 'FIT_001G_FLOW AT INLET HMMF-G', 'FIT_001H_FLOW AT INLET HMMF-H', 'FIT_001I_FLOW AT INLET HMMF-I', 'FIT_001J_FLOW AT INLET HMMF-J', 'FIT_003A_Cartridge_outflow', 'FIT_003B_

In [17]:
print("Chargement des données...")
dftime = pd.read_csv('data_evolution/df_minute_aggregated.csv')
df1["time"] = dftime["time"].values  # copie la colonne time
df1.set_index("time", inplace=True)  # remettre comme index

Chargement des données...


In [19]:
df1.head()

,INTAKE LINE PRESSURE OF HMMF FEED PUMPS,FIT_001A_FLOW AT INLET HMMF-A,FIT_001B_FLOW AT INLET HMMF-B,FIT_001C_FLOW AT INLET HMMF-C,FIT_001D_FLOW AT INLET HMMF-D,FIT_001E_FLOW AT INLET HMMF-E,FIT_001F_FLOW AT INLET HMMF-F,FIT_001G_FLOW AT INLET HMMF-G,FIT_001H_FLOW AT INLET HMMF-H,FIT_001I_FLOW AT INLET HMMF-I,...,PIT_007G_Discharge_pressure_HP PMP,PIT_007H_Discharge_pressure_HP PMP,PIT_008A_Reject line pressure,PIT_008B_Reject line pressure,PIT_008C_Reject line pressure,PIT_008D_Reject line pressure,PIT_008E_Reject line pressure,PIT_008F_Reject line pressure,PIT_008G_Reject line pressure,PIT_008H_Reject line pressure
time,,,,,,,,,,,,,,,,,,,,,
2024-02-13 11:38:00,1.49075,454.837494,391.059570,494.862335,388.456238,371.231262,452.521881,391.543762,452.603119,403.975006,...,60.618752,62.875000,61.412498,58.775002,61.956249,59.087502,61.381248,59.018749,58.3125,0.0
2024-02-13 11:39:00,1.49650,454.837494,388.655182,493.031921,388.496887,369.078125,452.521881,391.421875,451.587494,403.812500,...,60.618752,62.862499,61.368752,58.750000,61.987499,59.087502,61.531250,58.987499,58.3125,0.0
2024-02-13 11:40:00,1.49525,454.837494,386.984314,493.642059,388.984375,368.468750,452.521881,389.715637,449.596863,401.821869,...,60.618752,62.887501,61.393749,58.849998,61.943748,59.087502,61.443748,59.062500,58.3125,0.0
2024-02-13 11:41:00,1.49625,454.837494,386.413788,492.259064,386.181244,367.981262,452.521881,390.040619,450.165619,400.968750,...,60.618752,62.856251,61.299999,58.775002,61.924999,59.087502,61.556252,58.956249,58.3125,0.0
2024-02-13 11:42:00,1.50775,454.837494,387.391846,490.632050,387.968750,370.012512,452.521881,388.862488,450.409363,400.359375,...,60.618752,62.900002,61.162498,58.937500,61.924999,59.087502,61.556252,59.025002,58.3125,0.0


In [21]:


# Ajouter ces colonnes à df1
for col in cols_unique_df:
    if col in df.columns:  # pour éviter les erreurs si une colonne n'existe pas
        df1[col] = df[col]

# Vérification
print(df1.columns)


Index(['INTAKE LINE PRESSURE OF HMMF FEED PUMPS',
       'FIT_001A_FLOW AT INLET HMMF-A', 'FIT_001B_FLOW AT INLET HMMF-B',
       'FIT_001C_FLOW AT INLET HMMF-C', 'FIT_001D_FLOW AT INLET HMMF-D',
       'FIT_001E_FLOW AT INLET HMMF-E', 'FIT_001F_FLOW AT INLET HMMF-F',
       'FIT_001G_FLOW AT INLET HMMF-G', 'FIT_001H_FLOW AT INLET HMMF-H',
       'FIT_001I_FLOW AT INLET HMMF-I',
       ...
       'HMMF-I', 'HMMF-J', 'INDICE_DESEQUILIBRE_FILTRES',
       'INDICE_ENCRASSEMENT_HMMF', 'INDICE_IMPACT_PRODUCTION',
       'duree_backwash', 'freq_backwash', 'intervalle_min', 'target',
       'type_backwash'],
      dtype='object', length=104)


In [23]:
df1.head()

,INTAKE LINE PRESSURE OF HMMF FEED PUMPS,FIT_001A_FLOW AT INLET HMMF-A,FIT_001B_FLOW AT INLET HMMF-B,FIT_001C_FLOW AT INLET HMMF-C,FIT_001D_FLOW AT INLET HMMF-D,FIT_001E_FLOW AT INLET HMMF-E,FIT_001F_FLOW AT INLET HMMF-F,FIT_001G_FLOW AT INLET HMMF-G,FIT_001H_FLOW AT INLET HMMF-H,FIT_001I_FLOW AT INLET HMMF-I,...,HMMF-I,HMMF-J,INDICE_DESEQUILIBRE_FILTRES,INDICE_ENCRASSEMENT_HMMF,INDICE_IMPACT_PRODUCTION,duree_backwash,freq_backwash,intervalle_min,target,type_backwash
time,,,,,,,,,,,,,,,,,,,,,
2024-02-13 11:38:00,1.49075,454.837494,391.059570,494.862335,388.456238,371.231262,452.521881,391.543762,452.603119,403.975006,...,1,1,0.100190,0.695475,0.001675,271.0,734.0,NaN,1437.231247,3
2024-02-13 11:39:00,1.49650,454.837494,388.655182,493.031921,388.496887,369.078125,452.521881,391.421875,451.587494,403.812500,...,1,1,0.099537,0.694425,0.001676,271.0,734.0,1.0,1441.015625,3
2024-02-13 11:40:00,1.49525,454.837494,386.984314,493.642059,388.984375,368.468750,452.521881,389.715637,449.596863,401.821869,...,1,1,0.099028,0.693700,0.001677,271.0,734.0,1.0,1440.490616,3
2024-02-13 11:41:00,1.49625,454.837494,386.413788,492.259064,386.181244,367.981262,452.521881,390.040619,450.165619,400.968750,...,1,1,0.097855,0.692450,0.001677,271.0,734.0,1.0,1440.512512,3
2024-02-13 11:42:00,1.50775,454.837494,387.391846,490.632050,387.968750,370.012512,452.521881,388.862488,450.409363,400.359375,...,1,1,0.097786,0.693088,0.001678,271.0,734.0,1.0,1440.774994,3


In [25]:
df1.shape

(449002, 104)

In [27]:
print(df1.columns.tolist())

['INTAKE LINE PRESSURE OF HMMF FEED PUMPS', 'FIT_001A_FLOW AT INLET HMMF-A', 'FIT_001B_FLOW AT INLET HMMF-B', 'FIT_001C_FLOW AT INLET HMMF-C', 'FIT_001D_FLOW AT INLET HMMF-D', 'FIT_001E_FLOW AT INLET HMMF-E', 'FIT_001F_FLOW AT INLET HMMF-F', 'FIT_001G_FLOW AT INLET HMMF-G', 'FIT_001H_FLOW AT INLET HMMF-H', 'FIT_001I_FLOW AT INLET HMMF-I', 'FIT_001J_FLOW AT INLET HMMF-J', 'PDIT_001A_DIFF PRESSURE ACROSS HMMF-A', 'PDIT_001B_DIFF PRESSURE ACROSS HMMF-B', 'PDIT_001C_DIFF PRESSURE ACROSS HMMF-C', 'PDIT_001D_DIFF PRESSURE ACROSS HMMF-D', 'PDIT_001E_DIFF PRESSURE ACROSS HMMF-E', 'PDIT_001F_DIFF PRESSURE ACROSS HMMF-F', 'PDIT_001G_DIFF PRESSURE ACROSS HMMF-G', 'PDIT_001H_DIFF PRESSURE ACROSS HMMF-H', 'PDIT_001I_DIFF PRESSURE ACROSS HMMF-I', 'PDIT_001J_DIFF PRESSURE ACROSS HMMF-J', 'PIT_001 DISCHARGE PRESSURE AT HMMF A-E', 'PIT_001 DISCHARGE PRESSURE AT HMMF F-J', 'Niveau_reservoir', 'BACKWASH PUMP FLOW', 'LIT_003 OF HMMF BACKWASH STORAGE TANK', 'AIT007_PH at discharge HMMF FEED PUMPS', 'FIT_00

In [31]:
df1.shape

(449002, 104)

In [33]:
df1.index.name = "time"
df1.to_csv("dataset_finale.csv", index=True)

In [35]:
import joblib


# Charger le scaler du modèle
scaler = joblib.load('best_models/ML tank level/scaler_tuned.pkl')

# Colonnes utilisées par le modèle
cols_model = list(scaler.feature_names_in_)

# Afficher la liste
print(cols_model)


['INTAKE LINE PRESSURE OF HMMF FEED PUMPS', 'FIT_001A_FLOW AT INLET HMMF-A', 'FIT_001B_FLOW AT INLET HMMF-B', 'FIT_001C_FLOW AT INLET HMMF-C', 'FIT_001D_FLOW AT INLET HMMF-D', 'FIT_001E_FLOW AT INLET HMMF-E', 'FIT_001F_FLOW AT INLET HMMF-F', 'FIT_001G_FLOW AT INLET HMMF-G', 'FIT_001H_FLOW AT INLET HMMF-H', 'FIT_001I_FLOW AT INLET HMMF-I', 'FIT_001J_FLOW AT INLET HMMF-J', 'PDIT_001A_DIFF PRESSURE ACROSS HMMF-A', 'PDIT_001B_DIFF PRESSURE ACROSS HMMF-B', 'PDIT_001C_DIFF PRESSURE ACROSS HMMF-C', 'PDIT_001D_DIFF PRESSURE ACROSS HMMF-D', 'PDIT_001E_DIFF PRESSURE ACROSS HMMF-E', 'PDIT_001F_DIFF PRESSURE ACROSS HMMF-F', 'PDIT_001G_DIFF PRESSURE ACROSS HMMF-G', 'PDIT_001H_DIFF PRESSURE ACROSS HMMF-H', 'PDIT_001I_DIFF PRESSURE ACROSS HMMF-I', 'PDIT_001J_DIFF PRESSURE ACROSS HMMF-J', 'PIT_001 DISCHARGE PRESSURE AT HMMF A-E', 'PIT_001 DISCHARGE PRESSURE AT HMMF F-J']


C:\Users\yass3\anaconda3\envs\IDSI\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [22]:
# on recupere les liste de cols a utiliser dans chaque modele 

# ML tank level:
tank_list = ['INTAKE LINE PRESSURE OF HMMF FEED PUMPS', 'FIT_001A_FLOW AT INLET HMMF-A', 'FIT_001B_FLOW AT INLET HMMF-B', 
             'FIT_001C_FLOW AT INLET HMMF-C', 'FIT_001D_FLOW AT INLET HMMF-D', 'FIT_001E_FLOW AT INLET HMMF-E', 'FIT_001F_FLOW AT INLET HMMF-F', 
             'FIT_001G_FLOW AT INLET HMMF-G', 'FIT_001H_FLOW AT INLET HMMF-H', 'FIT_001I_FLOW AT INLET HMMF-I', 'FIT_001J_FLOW AT INLET HMMF-J',
             'PDIT_001A_DIFF PRESSURE ACROSS HMMF-A', 'PDIT_001B_DIFF PRESSURE ACROSS HMMF-B', 'PDIT_001C_DIFF PRESSURE ACROSS HMMF-C', 
             'PDIT_001D_DIFF PRESSURE ACROSS HMMF-D', 'PDIT_001E_DIFF PRESSURE ACROSS HMMF-E', 'PDIT_001F_DIFF PRESSURE ACROSS HMMF-F',
             'PDIT_001G_DIFF PRESSURE ACROSS HMMF-G', 'PDIT_001H_DIFF PRESSURE ACROSS HMMF-H', 'PDIT_001I_DIFF PRESSURE ACROSS HMMF-I',
             'PDIT_001J_DIFF PRESSURE ACROSS HMMF-J', 'PIT_001 DISCHARGE PRESSURE AT HMMF A-E', 'PIT_001 DISCHARGE PRESSURE AT HMMF F-J']

# ML prod finale:

features_list = [
    'Niveau_reservoir', 'DECISION_BACKWASH_NUM', 'DP_moyen_filtres', 'type_backwash',
    'duree_backwash', 'intervalle_min', 'freq_backwash', 'INDICE_ENCRASSEMENT_HMMF',
    'DEBIT_ENTREE_HMMF', 'INDICE_IMPACT_PRODUCTION', 'INDICE_DESEQUILIBRE_FILTRES',
    'PDIT_001A_DIFF PRESSURE ACROSS HMMF-A', 'PDIT_001B_DIFF PRESSURE ACROSS HMMF-B',
    'PDIT_001C_DIFF PRESSURE ACROSS HMMF-C', 'PDIT_001D_DIFF PRESSURE ACROSS HMMF-D',
    'PDIT_001E_DIFF PRESSURE ACROSS HMMF-E', 'PDIT_001F_DIFF PRESSURE ACROSS HMMF-F',
    'PDIT_001G_DIFF PRESSURE ACROSS HMMF-G', 'PDIT_001H_DIFF PRESSURE ACROSS HMMF-H',
    'PDIT_001I_DIFF PRESSURE ACROSS HMMF-I', 'PDIT_001J_DIFF PRESSURE ACROSS HMMF-J',
    'HMMF-A', 'HMMF-B', 'HMMF-C', 'HMMF-D', 'HMMF-E', 'HMMF-F', 'HMMF-G', 'HMMF-H',
    'HMMF-I', 'HMMF-J'
]



In [39]:
# Combiner toutes les colonnes à vérifier
all_features = tank_list + features_list

# Colonnes manquantes dans df1
missing_cols = [col for col in all_features if col not in df1.columns]

if not missing_cols:
    print("✅ Toutes les colonnes sont présentes dans df1.")
else:
    print("❌ Colonnes manquantes :", missing_cols)


✅ Toutes les colonnes sont présentes dans df1.


In [11]:
import pandas as pd
import joblib

# Fonction générique de prédiction
def predict_tank_and_prod(data_input, tank_list, features_list, tank_col_in_features='Niveau_reservoir'):
    """
    data_input : dict avec les valeurs d'entrée
    tank_list : colonnes pour le modèle tank
    features_list : colonnes pour le modèle final
    tank_col_in_features : nom de la colonne où injecter la prédiction du tank
    """
    # Charger les modèles et scalers
    tank_model = joblib.load('best_models/ML tank level/best_model_tuned.pkl')
    tank_scaler = joblib.load('best_models/ML tank level/scaler_tuned.pkl')
    prod_model = joblib.load('best_models/ML prod finale/model_Ridge_final.pkl')
    prod_scaler = joblib.load('best_models/ML prod finale/scaler_final.pkl')
    imputer = joblib.load('best_models/ML prod finale/imputer_final.pkl')
    
    # 1️⃣ Prédiction du tank
    data_tank = {col: data_input[col] for col in tank_list}
    X_tank = pd.DataFrame(data_tank)
    X_tank_scaled = tank_scaler.transform(X_tank)
    tank_pred = tank_model.predict(X_tank_scaled)[0]
    
    # 2️⃣ Injection dans le modèle final
    data_input[tank_col_in_features] = [tank_pred]
    X_prod = pd.DataFrame(data_input)
    X_prod = X_prod.reindex(columns=features_list)
    
    # 3️⃣ Imputation, standardisation et prédiction finale
    X_imputed = imputer.transform(X_prod)
    X_scaled = prod_scaler.transform(X_imputed)
    final_pred = prod_model.predict(X_scaled)[0]
    
    return tank_pred, final_pred


In [43]:
def extract_features_from_df(df, index, tank_list, features_list, tank_col='target'):
    """
    df : DataFrame complet (df1)
    index : indice de la ligne à utiliser
    tank_list : colonnes pour le modèle tank
    features_list : colonnes pour le modèle final
    tank_col : colonne cible du tank à exclure pour la sélection
    """
    # Extraire la ligne
    row = df.iloc[index]
    
    # Sélectionner les colonnes tank et features (sans la colonne target/tank)
    selected_cols = tank_list + [col for col in features_list if col != tank_col]
    
    # Créer dictionnaire avec valeurs sous forme de liste pour Pandas DataFrame
    data_dict = {col: [row[col]] for col in selected_cols if col in df.columns}
    
    return data_dict

# Exemple d'utilisation
data_selected = extract_features_from_df(df1, 0, tank_list, features_list)
print(data_selected)


{'INTAKE LINE PRESSURE OF HMMF FEED PUMPS': [np.float64(1.4907499551773071)], 'FIT_001A_FLOW AT INLET HMMF-A': [np.float64(454.8374938964844)], 'FIT_001B_FLOW AT INLET HMMF-B': [np.float64(391.0595703125)], 'FIT_001C_FLOW AT INLET HMMF-C': [np.float64(494.8623352050781)], 'FIT_001D_FLOW AT INLET HMMF-D': [np.float64(388.4562377929688)], 'FIT_001E_FLOW AT INLET HMMF-E': [np.float64(371.2312622070313)], 'FIT_001F_FLOW AT INLET HMMF-F': [np.float64(452.5218811035156)], 'FIT_001G_FLOW AT INLET HMMF-G': [np.float64(391.5437622070313)], 'FIT_001H_FLOW AT INLET HMMF-H': [np.float64(452.6031188964844)], 'FIT_001I_FLOW AT INLET HMMF-I': [np.float64(403.9750061035156)], 'FIT_001J_FLOW AT INLET HMMF-J': [np.float64(352.21875)], 'PDIT_001A_DIFF PRESSURE ACROSS HMMF-A': [np.float64(0.6065000295639038)], 'PDIT_001B_DIFF PRESSURE ACROSS HMMF-B': [np.float64(0.8165000081062317)], 'PDIT_001C_DIFF PRESSURE ACROSS HMMF-C': [np.float64(0.6602500081062317)], 'PDIT_001D_DIFF PRESSURE ACROSS HMMF-D': [np.flo

In [45]:

# Exemple d'utilisation
tank_val, prod_val = predict_tank_and_prod(data_selected, tank_list, features_list)
print("Tank prédit :", tank_val)
print("Production finale :", prod_val)


C:\Users\yass3\anaconda3\envs\IDSI\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator KNeighborsRegressor from version 1.2.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\yass3\anaconda3\envs\IDSI\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\yass3\anaconda3\envs\IDSI\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Ridge from version 1.6.1 when using version 1.7.2. T

Tank prédit : 82.89375305175781
Production finale : 1291.306902066573


In [49]:
df1[['Niveau_reservoir', 'target']].head()


,Niveau_reservoir,target
time,,
2024-02-13 11:38:00,82.893753,1437.231247
2024-02-13 11:39:00,82.375000,1441.015625
2024-02-13 11:40:00,81.356247,1440.490616
2024-02-13 11:41:00,81.487503,1440.512512
2024-02-13 11:42:00,81.237503,1440.774994


In [51]:
def test(nombre):
    data_selected = extract_features_from_df(df1, nombre, tank_list, features_list)

    # Exemple d'utilisation
    tank_val, prod_val = predict_tank_and_prod(data_selected, tank_list, features_list)
    print("Tank prédit :", tank_val)
    print("Production finale :", prod_val)


In [55]:
test(6)

Tank prédit : 81.9749984741211
Production finale : 1244.9610057070602


C:\Users\yass3\anaconda3\envs\IDSI\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator KNeighborsRegressor from version 1.2.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\yass3\anaconda3\envs\IDSI\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\yass3\anaconda3\envs\IDSI\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Ridge from version 1.6.1 when using version 1.7.2. T

In [57]:
df1['target'].max()


np.float64(1700.6499938964844)

In [24]:
import itertools
import pandas as pd
import numpy as np

# ✅ Règles pour chaque feature
feature_rules = {
    # Variables continues avec seuils
    'Niveau_reservoir': {'type': 'float', 'min': 0, 'max': 100, 'default': 50},
    'hmmf_inlet_flow': {'type': 'float', 'min': 200, 'max': 650, 'default': 400},
    'ro_permeate_flow': {'type': 'float', 'min': 50, 'max': 250, 'default': 150},
    'hmmf_diff_pressure': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},

    # Variables de backwash
    'DECISION_BACKWASH_NUM': {'type': 'int', 'allowed': [0, 1, 2, 3]},
    'duree_backwash': {'type': 'float', 'min': 1, 'max': 60, 'default': 10},
    'intervalle_min': {'type': 'float', 'min': 0, 'max': 60, 'default': 60},
    'freq_backwash': {'type': 'float', 'min': 0, 'max': 10, 'default': 1},

    # Indices calculés
    'INDICE_ENCRASSEMENT_HMMF': {'type': 'float', 'min': 0, 'max': 1, 'default': 0},
    'DEBIT_ENTREE_HMMF': {'type': 'float', 'min': 100, 'max': 600, 'default': 200},
    'INDICE_IMPACT_PRODUCTION': {
        'type': 'float',
        'calculation': lambda df: df["INDICE_ENCRASSEMENT_HMMF"] / (df["DEBIT_ENTREE_HMMF"] + 1e-6)
    },
    'INDICE_DESEQUILIBRE_FILTRES': {'type': 'float', 'min': 0, 'max': 1, 'default': 0},
    'DP_moyen_filtres': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},

    # HMMF filters (binaire 0/1)
    'HMMF-A': {'type': 'int', 'allowed': [0,1]},
    'HMMF-B': {'type': 'int', 'allowed': [0,1]},
    'HMMF-C': {'type': 'int', 'allowed': [0,1]},
    'HMMF-D': {'type': 'int', 'allowed': [0,1]},
    'HMMF-E': {'type': 'int', 'allowed': [0,1]},
    'HMMF-F': {'type': 'int', 'allowed': [0,1]},
    'HMMF-G': {'type': 'int', 'allowed': [0,1]},
    'HMMF-H': {'type': 'int', 'allowed': [0,1]},
    'HMMF-I': {'type': 'int', 'allowed': [0,1]},
    'HMMF-J': {'type': 'int', 'allowed': [0,1]},

    # Pressions moyennes des filtres HMMF (DP)
    'PDIT_001A_DIFF PRESSURE ACROSS HMMF-A': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},
    'PDIT_001B_DIFF PRESSURE ACROSS HMMF-B': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},
    'PDIT_001C_DIFF PRESSURE ACROSS HMMF-C': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},
    'PDIT_001D_DIFF PRESSURE ACROSS HMMF-D': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},
    'PDIT_001E_DIFF PRESSURE ACROSS HMMF-E': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},
    'PDIT_001F_DIFF PRESSURE ACROSS HMMF-F': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},
    'PDIT_001G_DIFF PRESSURE ACROSS HMMF-G': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},
    'PDIT_001H_DIFF PRESSURE ACROSS HMMF-H': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},
    'PDIT_001I_DIFF PRESSURE ACROSS HMMF-I': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},
    'PDIT_001J_DIFF PRESSURE ACROSS HMMF-J': {'type': 'float', 'min': 0, 'max': 1.5, 'default': 0.5},
}

# ✅ Fonction de validation d'une ligne
def validate_row(row, rules=feature_rules):
    for col, rule in rules.items():
        if col not in row:
            continue
        val = row[col]
        if 'allowed' in rule and val not in rule['allowed']:
            return False
        if 'min' in rule and val < rule['min']:
            return False
        if 'max' in rule and val > rule['max']:
            return False
    return True

# --- Générateur complet avec variation pour colonnes continues ---
def generate_all_combinations(rules=feature_rules, features_list=None, num_points=3):
    categorical_cols = []
    continuous_cols = []

    # Séparer colonnes catégorielles et continues
    for col, rule in rules.items():
        if 'allowed' in rule:
            categorical_cols.append((col, rule['allowed']))
        elif 'min' in rule and 'max' in rule:
            continuous_cols.append((col, rule))

    # Combinaisons pour colonnes catégorielles
    cat_names = [col for col, _ in categorical_cols]
    cat_values = [vals for _, vals in categorical_cols]
    cat_combinations = list(itertools.product(*cat_values))
    df_cat = pd.DataFrame(cat_combinations, columns=cat_names)

    # Générer valeurs multiples pour colonnes continues
    for col, rule in continuous_cols:
        vals = np.linspace(rule['min'], rule['max'], num=num_points)
        # répéter pour correspondre à df_cat
        df_cat[col] = np.tile(vals, len(df_cat) // len(vals) + 1)[:len(df_cat)]

    # Ajouter toutes les colonnes manquantes dans features_list
    if features_list:
        for col in features_list:
            if col not in df_cat.columns:
                df_cat[col] = 0

    # Calculer indices calculés
    if 'INDICE_IMPACT_PRODUCTION' in df_cat.columns:
        df_cat['INDICE_IMPACT_PRODUCTION'] = df_cat['INDICE_ENCRASSEMENT_HMMF'] / (
            df_cat['DEBIT_ENTREE_HMMF'] + 1e-6
        )

    # Réordonner selon features_list si fourni
    if features_list:
        df_cat = df_cat[features_list]

    return df_cat

# --- Exemple d'utilisation ---
features_list = list(feature_rules.keys())
all_combos = generate_all_combinations(features_list=features_list, num_points=5)
print("Nombre de combinaisons générées :", len(all_combos))
all_combos.to_csv("all_combinations_pipeline.csv", index=False)


Nombre de combinaisons générées : 4096


✅ Ce que ça fait :

Parcourt toutes les colonnes avec allowed → crée toutes les combinaisons possibles.

Remplit les colonnes continues avec leur valeur par défaut.

Calcule automatiquement INDICE_IMPACT_PRODUCTION.

Renvoie un DataFrame prêt à tester sur ton modèle.

In [26]:

# Colonnes tank à ajouter avec leur "moyenne" calculée à partir des bornes
tank_cols_means = {
    'INTAKE LINE PRESSURE OF HMMF FEED PUMPS': (0.5 + 2.5)/2,
    'FIT_001A_FLOW AT INLET HMMF-A': (200 + 650)/2,
    'FIT_001B_FLOW AT INLET HMMF-B': (200 + 650)/2,
    'FIT_001C_FLOW AT INLET HMMF-C': (200 + 650)/2,
    'FIT_001D_FLOW AT INLET HMMF-D': (200 + 650)/2,
    'FIT_001E_FLOW AT INLET HMMF-E': (200 + 650)/2,
    'FIT_001F_FLOW AT INLET HMMF-F': (200 + 650)/2,
    'FIT_001G_FLOW AT INLET HMMF-G': (200 + 650)/2,
    'FIT_001H_FLOW AT INLET HMMF-H': (200 + 650)/2,
    'FIT_001I_FLOW AT INLET HMMF-I': (200 + 650)/2,
    'FIT_001J_FLOW AT INLET HMMF-J': (200 + 650)/2,
    'PIT_001 DISCHARGE PRESSURE AT HMMF A-E': (1 + 3.5)/2,
    'PIT_001 DISCHARGE PRESSURE AT HMMF F-J': (1 + 3.5)/2
}

# Ajouter ou remplacer les colonnes dans le DataFrame
for col, mean_val in tank_cols_means.items():
    all_combos[col] = mean_val

# Vérification
print(all_combos[tank_cols_means.keys()].head())

   INTAKE LINE PRESSURE OF HMMF FEED PUMPS  FIT_001A_FLOW AT INLET HMMF-A  \
0                                      1.5                          425.0   
1                                      1.5                          425.0   
2                                      1.5                          425.0   
3                                      1.5                          425.0   
4                                      1.5                          425.0   

   FIT_001B_FLOW AT INLET HMMF-B  FIT_001C_FLOW AT INLET HMMF-C  \
0                          425.0                          425.0   
1                          425.0                          425.0   
2                          425.0                          425.0   
3                          425.0                          425.0   
4                          425.0                          425.0   

   FIT_001D_FLOW AT INLET HMMF-D  FIT_001E_FLOW AT INLET HMMF-E  \
0                          425.0                          425.0   


In [27]:
all_combos.to_csv("all_combinations_pipeline.csv", index=False)
